# Qwen2.5-1.5B Ablation v1 — 官方 8-shot GSM8K

**本 notebook 为修复版（v1），与旧版完全隔离，结果存 `logs/v1/`**

| Phase | 内容 | 预计耗时 |
|-------|------|----------|
| **P0** | 环境 + Drive + 代码同步 | ~5 min |
| **P1a** | Group B SFT GSM8K（8-shot, n=500，产 badcase） | ~2h |
| **P2** | 错误分类 + Targeted DPO 数据生成 | ~45 min |
| **P4** | Group D 训练 + 评测（GSM8K, n=300） | ~1.5h |
| **P1b** | Group B DPO 评测（GSM8K, n=300） | ~1h |
| **P3** | Group A 训练 + 评测（GSM8K, n=300） | ~4.5h |
| **P5** | 汇总 | ~2 min |

**修复清单（相对旧版）**:
- ✅ `gsm8k_cot_zeroshot` → `gsm8k_cot`（8-shot，与 Qwen 官方一致）
- ✅ `ignore_mismatched_sizes=True`（修复 NF4 模型加载失败）
- ✅ lm-eval 结果路径解析修复（`results_*.json` 递归查找）
- ✅ P1a 增至 n=500（获得更多 badcase，目标 ~175 条）
- ✅ 所有结果写 `logs/v1/`，不读旧文件
- ✅ 不跑 MATH（用 logs 2/ 已有自定义协议结果，报告中注明差异）

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P0.1: 安装依赖 + GPU 检测 + V1 路径常量 + 工具函数
# ═══════════════════════════════════════════════════════════════════

# ── 安装（固定版本防止冲突）──────────────────────────────────────
!pip install -q -U pip
!pip install -q "unsloth[colab-new]>=2025.1.0" "trl>=0.14.0" "peft>=0.14.0" \
    "bitsandbytes>=0.45.0" "transformers>=4.49.0" "datasets>=3.2.0" \
    "accelerate>=1.2.0" "pyyaml>=6.0.2" "safetensors" "tqdm" "scipy" \
    "lm-eval[api]>=0.4.5"

import torch, os, subprocess, json, sys, glob, shutil, yaml, math
from pathlib import Path
from datetime import datetime
from collections import Counter

assert torch.cuda.is_available(), '❌ 未检测到 GPU，请切换到 A100 Runtime'
gpu = torch.cuda.get_device_properties(0)
print(f'✅ PyTorch {torch.__version__} | GPU: {gpu.name} | VRAM: {gpu.total_memory/1e9:.1f} GB')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ── V1 路径常量（完全隔离，不读旧 logs/ 结果）───────────────────
V1            = 'logs/v1'
V1_GSM_B_SFT  = f'{V1}/gsm8k_b_sft.json'
V1_GSM_B_DPO  = f'{V1}/gsm8k_b_dpo.json'
V1_GSM_D      = f'{V1}/gsm8k_d.json'
V1_GSM_A_SFT  = f'{V1}/gsm8k_a_sft.json'
V1_GSM_A_DPO  = f'{V1}/gsm8k_a_dpo.json'
V1_BC         = f'{V1}/gsm8k_b_sft_badcases.jsonl'
V1_DPO_DATA   = 'data/processed/dpo_targeted_v1.json'
V1_GD_DPO     = 'outputs/group_d_v1/dpo'
V1_GD_MERGED  = 'outputs/group_d_v1/merged'

# ── 工具函数 ─────────────────────────────────────────────────────
def stream_run(cmd, log_path=None):
    '''实时流式打印子进程输出（无缓冲），可选写入日志。返回 returncode。'''
    if cmd[0] == 'python3' and '-u' not in cmd:
        cmd = ['python3', '-u'] + cmd[1:]
    log_f = open(log_path, 'a', encoding='utf-8') if log_path else None
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
        if log_f:
            log_f.write(line); log_f.flush()
    proc.wait()
    if log_f: log_f.close()
    return proc.returncode


def run_lm_eval_gsm8k(model_path, output_json, limit=300, badcase_output=''):
    '''GSM8K 8-shot 评测（via lm_eval_run.py，已修复协议 + NF4）。
    ✅ 断点续跑：output_json 已存在且 total>=limit*0.9 则跳过。
    ✅ 路径隔离：output_json 在 logs/v1/，不读旧文件。
    '''
    Path(output_json).parent.mkdir(parents=True, exist_ok=True)
    if Path(output_json).exists():
        try:
            d = json.loads(Path(output_json).read_text())
            if d.get('total', 0) >= int(limit * 0.9):
                print(f'  ✅ 已有结果: {d.get("accuracy", 0):.1%} ({d.get("total", 0)}题) → {output_json}')
                return True
        except Exception:
            pass
    cmd = ['python3', '-u', 'eval/lm_eval_run.py',
           '--model_path', model_path,
           '--tasks', 'gsm8k',
           '--limit', str(limit),
           '--output', output_json]
    if badcase_output:
        Path(badcase_output).parent.mkdir(parents=True, exist_ok=True)
        cmd += ['--badcase_output', badcase_output]
    print(f'  GSM8K 8-shot (n={limit}) → {output_json}')
    rc = stream_run(cmd)
    if rc != 0:
        print(f'  ❌ lm_eval 失败 (exit {rc})')
        return False
    if Path(output_json).exists():
        d = json.loads(Path(output_json).read_text())
        print(f'  📊 GSM8K: {d.get("accuracy", 0):.1%} ({d.get("correct", 0)}/{d.get("total", 0)})')
    return True


def ensure_fp16_merged(merged_path, adapter_hint='', label=''):
    '''检测 NF4 量化模型，尝试重合并为 fp16。
    lm_eval_run.py 已设 ignore_mismatched_sizes=True 作为兜底。
    '''
    if not merged_path or not os.path.isfile(f'{merged_path}/config.json'):
        return merged_path
    try:
        cfg = json.load(open(f'{merged_path}/config.json'))
        if cfg.get('quantization_config', {}).get('quant_type', '') != 'nf4':
            return merged_path
    except Exception:
        return merged_path
    fp16_path = merged_path.rstrip('/') + '_fp16'
    if os.path.isfile(f'{fp16_path}/config.json'):
        print(f'  {label}: fp16 版已存在 → {fp16_path}')
        return fp16_path
    adapter = adapter_hint
    if not adapter or not os.path.isfile(f'{adapter}/adapter_config.json'):
        print(f'  {label}: NF4 检测到，ignore_mismatched_sizes=True 兜底')
        return merged_path
    print(f'  {label}: NF4 检测到，重合并为 fp16 → {fp16_path}')
    r = subprocess.run(['python3', 'scripts/merge_lora.py',
                        '--adapter_path', adapter, '--output_path', fp16_path],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print(f'  ✅ {label} fp16 完成')
        return fp16_path
    print(f'  ⚠️ 重合并失败，使用原路径 + ignore_mismatched_sizes')
    return merged_path

print('✅ 工具函数已定义')
print(f'📁 V1 结果目录: {V1}  (完全隔离，不读旧 logs/lm_*.json)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P0.2: 挂载 Drive + 设置 PROJECT_DIR + 读取密钥
# ═══════════════════════════════════════════════════════════════════
from google.colab import drive, userdata

drive.mount('/content/drive')

DRIVE_DIR   = '/content/drive/MyDrive/Qwen-Reasoning'
PROJECT_DIR = DRIVE_DIR
os.chdir(PROJECT_DIR)
print(f'工作目录: {PROJECT_DIR}')

for env_key, secret_key in [('HF_TOKEN', 'HF_TOKEN'),
                             ('HUGGING_FACE_HUB_TOKEN', 'HF_TOKEN'),
                             ('DASHSCOPE_API_KEY', 'DASHSCOPE_API_KEY')]:
    try:
        os.environ[env_key] = userdata.get(secret_key)
        print(f'  {env_key}: OK')
    except Exception:
        print(f'  {env_key}: missing')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

for d in [V1, 'data/processed', 'outputs', 'config',
           'results/errors/sft_v1/by_type', 'results/ablation']:
    os.makedirs(d, exist_ok=True)

print('✅ Drive 挂载完成，目录已就绪')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P0.3: 同步代码（GitHub → Drive）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)
sys.path.insert(0, f'{PROJECT_DIR}/eval')
sys.path.insert(0, f'{PROJECT_DIR}/scripts')

if os.path.isdir(f'{PROJECT_DIR}/.git'):
    r = subprocess.run(['git', 'pull', '--rebase'], capture_output=True, text=True)
    print('git pull:', r.stdout.strip() or r.stderr.strip() or 'already up-to-date')
else:
    tmp = '/tmp/_repo_tmp'
    if os.path.isdir(tmp): shutil.rmtree(tmp)
    r = subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/yukiiii0730/6000Q-QwenMiniReason.git', tmp],
        capture_output=True, text=True)
    if r.returncode == 0:
        for item in os.listdir(tmp):
            if item.startswith('.'): continue
            src_p, dst_p = os.path.join(tmp, item), os.path.join(PROJECT_DIR, item)
            if os.path.isdir(src_p): shutil.copytree(src_p, dst_p, dirs_exist_ok=True)
            else: shutil.copy2(src_p, dst_p)
        shutil.rmtree(tmp)
        print('✅ 代码已从 GitHub 同步')
    else:
        print(f'⚠️ clone 失败，使用 Drive 中现有代码\n{r.stderr[:300]}')

# 验证关键脚本
ok = True
for f in ['eval/lm_eval_run.py', 'scripts/dpo_train.py', 'scripts/merge_lora.py',
          'scripts/classify_errors.py', 'scripts/build_targeted_dpo.py',
          'scripts/sft_train.py']:
    status = '✅' if os.path.isfile(f) else '❌ MISSING'
    if '❌' in status: ok = False
    print(f'  {status} {f}')

if not ok:
    raise RuntimeError('关键脚本缺失，请检查 GitHub 同步是否正常')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P0.4: 检查模型路径 + NF4 处理 + 自适应 batch size
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

# ── 自动发现 Group B 模型 ──────────────────────────────────────
SFT_MERGED   = None   # Group B SFT (fp16 优先)
SFT_ADAPTER  = None   # Group B SFT LoRA adapter
DPO_MERGED   = None   # Group B DPO (fp16 优先)
DPO_ADAPTER  = None   # Group B DPO LoRA adapter

for p in ['outputs/sft_merged_fp16', 'outputs/sft_merged']:
    if os.path.isfile(f'{p}/config.json'): SFT_MERGED = p; break
for p in ['outputs/merged_fp16', 'outputs/merged']:
    if os.path.isfile(f'{p}/config.json'): DPO_MERGED = p; break
for p in ['outputs/sft']:
    if os.path.isfile(f'{p}/adapter_config.json'): SFT_ADAPTER = p; break
for p in ['outputs/dpo']:
    if os.path.isfile(f'{p}/adapter_config.json'): DPO_ADAPTER = p; break

print('📋 原始模型状态:')
print(f'  Group B SFT merged : {SFT_MERGED or "MISSING"}')
print(f'  Group B DPO merged : {DPO_MERGED or "MISSING"}')
print(f'  Group B SFT adapter: {SFT_ADAPTER or "not found"}')
print(f'  Group B DPO adapter: {DPO_ADAPTER or "not found"}')

# NF4 → fp16 检测（lm_eval_run.py 有 ignore_mismatched_sizes=True 兜底）
if SFT_MERGED:
    SFT_MERGED = ensure_fp16_merged(SFT_MERGED, SFT_ADAPTER or 'outputs/sft', 'Group B SFT')
if DPO_MERGED:
    DPO_MERGED = ensure_fp16_merged(DPO_MERGED, DPO_ADAPTER or 'outputs/dpo', 'Group B DPO')

print(f'\n→ P1a 评测将使用: {SFT_MERGED}')
print(f'→ P1b 评测将使用: {DPO_MERGED}')

# ── 数据文件检查 ────────────────────────────────────────────────
print('\n📋 训练数据:')
for fn in ['sft_train.json', 'dpo_train.json', 'dpo_targeted_v1.json']:
    fp = f'data/processed/{fn}'
    sz = f'{os.path.getsize(fp)/1e6:.1f} MB' if os.path.isfile(fp) else 'MISSING'
    print(f'  {fn:35s}: {sz}')

# ── Batch size 自适应 ───────────────────────────────────────────
vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH_SIZE = 4 if vram_gb > 70 else 2
print(f'\n⚙️  BATCH_SIZE={BATCH_SIZE}, grad_accum=8, eff_BS={BATCH_SIZE*8}')

if not SFT_MERGED:
    print('\n❌ Group B SFT 缺失，P1a/P2/P4 无法执行！')
else:
    print('\n✅ 准备就绪')

---
## P1a: Group B SFT GSM8K 评测（8-shot, n=500）

- 评测协议：`gsm8k_cot`（8-shot，与 Qwen 官方一致）
- n=500：正确率约 65% → ~175 badcase，比上版 77 条多 2×
- 结果写 `logs/v1/gsm8k_b_sft.json`，不读旧文件
- **预计耗时：~2h**

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P1a: Group B SFT GSM8K 评测（8-shot, n=500）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

if not SFT_MERGED:
    print('⚠️ Group B SFT 不存在，跳过 P1a')
else:
    print('▶ P1a: Group B SFT GSM8K (8-shot, n=500)')
    ok = run_lm_eval_gsm8k(SFT_MERGED, V1_GSM_B_SFT, limit=500, badcase_output=V1_BC)
    if ok and Path(V1_BC).exists():
        n_bc = sum(1 for _ in open(V1_BC))
        print(f'\n  Badcase: {n_bc} 条 → {V1_BC}')
        if n_bc < 30:
            print('  ⚠️ badcase 太少，P2 生成的 DPO 数据质量可能偏低')
        elif n_bc > 150:
            print('  ✅ badcase 充足，P2 可生成高质量 Targeted DPO 数据')

print('\nP1a 完成')

---
## P2: 错误分类 + Targeted DPO 数据生成

- L3: `classify_errors.py`（qwen-flash）→ 5 类错误分桶
- L4: `build_targeted_dpo.py`（qwen3-235b）→ 类型专属 DPO 对
- 结果写 `data/processed/dpo_targeted_v1.json`（v1 专属）
- **需要 DASHSCOPE_API_KEY**
- **预计耗时：~45 min**

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P2: 错误分类（L3）+ Targeted DPO 数据生成（L4）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

# 断点续跑：v1 数据已存在则跳过
if Path(V1_DPO_DATA).exists():
    data = json.load(open(V1_DPO_DATA))
    print(f'✅ dpo_targeted_v1.json 已存在 ({len(data)} 条)，跳过 P2')
    types = Counter(d.get('error_type', '?') for d in data)
    for t, n in types.most_common(): print(f'  {t}: {n}')
else:
    api_key = os.environ.get('DASHSCOPE_API_KEY', '')
    if not api_key:
        print('⚠️ DASHSCOPE_API_KEY 未设置，跳过 P2')
        print('   请本地运行后上传 data/processed/dpo_targeted_v1.json 到 Drive')
    elif not Path(V1_BC).exists():
        print(f'❌ Badcase 文件不存在: {V1_BC}，请先完成 P1a')
    else:
        n_bc = sum(1 for _ in open(V1_BC))
        print(f'▶ P2: 对 {n_bc} 条 badcase 分类 + 生成 Targeted DPO 数据')

        # L3: 错误分类
        print('\n── L3: 错误分类（qwen-flash）──')
        rc = stream_run(['python3', 'scripts/classify_errors.py',
                         '--badcase_jsonl', V1_BC,
                         '--output_dir', 'results/errors/sft_v1',
                         '--workers', '4'])
        if rc != 0:
            print('❌ classify_errors.py 失败')
        else:
            print('✅ L3 完成，分类分布:')
            for f in sorted(glob.glob('results/errors/sft_v1/by_type/*.jsonl')):
                cnt = sum(1 for _ in open(f))
                print(f'  {Path(f).stem}: {cnt} 条')

            # L4: 生成 Targeted DPO 数据
            print('\n── L4: 生成 Targeted DPO（qwen3-235b）──')
            rc = stream_run(['python3', 'scripts/build_targeted_dpo.py',
                             '--by_type_dir', 'results/errors/sft_v1/by_type',
                             '--per_type_n', '200',
                             '--tag', 'v1',
                             '--output_dir', 'data/processed',
                             '--workers', '4'])
            if rc != 0:
                print('❌ build_targeted_dpo.py 失败')
            elif Path(V1_DPO_DATA).exists():
                data = json.load(open(V1_DPO_DATA))
                print(f'✅ L4 完成: {len(data)} 条 → {V1_DPO_DATA}')
                types = Counter(d.get('error_type', '?') for d in data)
                for t, n in types.most_common(): print(f'  {t}: {n}')

print('\nP2 完成')

---
## P4: Group D 训练（Error-Type-Targeted DPO）

- 基座：Group B SFT merged
- 数据：`dpo_targeted_v1.json`（v1 专属）
- 模型输出：`outputs/group_d_v1/`（不覆盖旧版）
- 评测：GSM8K 8-shot, n=300
- **预计耗时：~1.5h**

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P4: Group D 训练（Error-Type-Targeted DPO，v1 版）
# 输出到 outputs/group_d_v1/（不覆盖旧版 outputs/group_d/ 或 group_d_lm/）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

if not SFT_MERGED:
    print('❌ Group B SFT 模型不存在，跳过 P4')
elif not Path(V1_DPO_DATA).exists():
    print(f'❌ {V1_DPO_DATA} 不存在，请先完成 P2')
elif Path(f'{V1_GD_MERGED}/config.json').exists():
    print(f'✅ Group D v1 merged 已存在，跳过训练 → {V1_GD_MERGED}')
else:
    data = json.load(open(V1_DPO_DATA))
    n_data = len(data)
    print(f'▶ P4: Group D Targeted DPO 训练 ({n_data} 条)')
    types = Counter(d.get('error_type', '?') for d in data)
    for t, n in types.most_common(): print(f'  {t}: {n}')

    # 自适应步数
    ERROR_WEIGHTS = {'arithmetic': 1, 'reasoning_skip': 2, 'setup_error': 3,
                     'unit_or_format': 1, 'extraction_error': 1}
    weighted_n = sum(max(1, math.ceil(ERROR_WEIGHTS.get(d.get('error_type',''), 1)))
                     for d in data)
    eff_bs      = BATCH_SIZE * 8
    steps_ep    = max(1, weighted_n // eff_bs)
    max_steps   = max(30, steps_ep * 5)
    warmup      = max(5,  max_steps // 10)
    save_steps  = max(10, max_steps // 5)
    print(f'\n  加权后: {weighted_n} 条 | {steps_ep} steps/ep × 5 = {max_steps} steps')
    print(f'  warmup={warmup}, save_every={save_steps}')

    # 断点续跑：检测已有 checkpoint
    ckpts = sorted(glob.glob(f'{V1_GD_DPO}/checkpoint-*'))
    if ckpts: print(f'\n♻️  续跑: {ckpts[-1]}')

    # 写配置
    dpo_cfg = {
        'model_name': 'Qwen/Qwen2.5-1.5B-Instruct',
        'base_adapter_path': SFT_MERGED,
        'output_dir': V1_GD_DPO,
        'max_seq_length': 2048, 'load_in_4bit': True, 'seed': 42,
        'beta': 0.1, 'loss_type': 'sigmoid',
        'error_type_weights': ERROR_WEIGHTS,
        'dataset': {'name': 'local', 'split': 'train', 'max_samples': -1},
        'lora': {'use_dora': True, 'r': 16, 'alpha': 32, 'dropout': 0.0,
                 'target_modules': ['q_proj','k_proj','v_proj','o_proj',
                                    'gate_proj','up_proj','down_proj']},
        'train': {
            'per_device_train_batch_size': BATCH_SIZE,
            'gradient_accumulation_steps': 8,
            'warmup_steps': warmup, 'max_steps': max_steps, 'learning_rate': 1e-5,
            'logging_steps': 5, 'save_steps': save_steps, 'weight_decay': 0.0,
            'lr_scheduler_type': 'cosine', 'optim': 'paged_adamw_8bit',
            'fp16': False, 'bf16': True, 'dataloader_num_workers': 4,
        },
        'dataset_path': V1_DPO_DATA,
    }
    os.makedirs('config', exist_ok=True)
    with open('config/dpo_config_group_d_v1.yaml', 'w') as f:
        yaml.dump(dpo_cfg, f, allow_unicode=True)

    # 训练（实时输出 + 写日志）
    ts       = datetime.now().strftime('%Y%m%d_%H%M%S')
    log_dir  = f'logs/runs/{ts}_group_d_v1_dpo'
    os.makedirs(log_dir, exist_ok=True)
    log_path = f'{log_dir}/train.log'
    print(f'\n训练日志: {log_path}')
    rc = stream_run(['python3', 'scripts/dpo_train.py',
                     '--config', 'config/dpo_config_group_d_v1.yaml'],
                    log_path=log_path)
    if rc != 0:
        raise RuntimeError(f'Group D DPO 训练失败 (exit {rc})，日志: {log_path}')

    # 合并 adapter → fp16
    print(f'\n合并 Group D v1 adapter → {V1_GD_MERGED}')
    os.makedirs(V1_GD_MERGED, exist_ok=True)
    subprocess.run(['python3', 'scripts/merge_lora.py',
                    '--adapter_path', V1_GD_DPO,
                    '--base_model', SFT_MERGED,
                    '--output_path', V1_GD_MERGED], check=True)
    print('✅ Group D v1 训练 + 合并完成')

print('\nP4 完成')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P4.2: Group D v1 评测（GSM8K 8-shot, n=300）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

if not Path(f'{V1_GD_MERGED}/config.json').exists():
    print(f'⚠️ Group D v1 模型不存在，跳过')
else:
    print('▶ P4.2: Group D v1 GSM8K (8-shot, n=300)')
    run_lm_eval_gsm8k(V1_GD_MERGED, V1_GSM_D, limit=300)

    # 与 B SFT 对比
    if Path(V1_GSM_B_SFT).exists() and Path(V1_GSM_D).exists():
        b_acc = json.loads(Path(V1_GSM_B_SFT).read_text()).get('accuracy', 0)
        d_acc = json.loads(Path(V1_GSM_D).read_text()).get('accuracy', 0)
        print(f'\n  📊 Targeted DPO 效果: {d_acc-b_acc:+.1%} (D vs B-SFT)')
        print(f'     B-SFT={b_acc:.1%}  D={d_acc:.1%}')

print('\nP4.2 完成')

---
## P1b: Group B DPO 评测（GSM8K 8-shot, n=300）

- 不跑 MATH（报告中用 logs 2/ 自定义协议结果）
- **预计耗时：~1h**

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P1b: Group B DPO 评测（GSM8K 8-shot, n=300）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

if not DPO_MERGED:
    print('⚠️ Group B DPO 模型不存在，跳过 P1b')
else:
    print('▶ P1b: Group B DPO GSM8K (8-shot, n=300)')
    run_lm_eval_gsm8k(DPO_MERGED, V1_GSM_B_DPO, limit=300)

    # Standard DPO 效果对比
    if Path(V1_GSM_B_SFT).exists() and Path(V1_GSM_B_DPO).exists():
        sft_acc = json.loads(Path(V1_GSM_B_SFT).read_text()).get('accuracy', 0)
        dpo_acc = json.loads(Path(V1_GSM_B_DPO).read_text()).get('accuracy', 0)
        print(f'\n  📊 Standard DPO 效果: {dpo_acc-sft_acc:+.1%} (B-DPO vs B-SFT)')
        print(f'     B-SFT={sft_acc:.1%}  B-DPO={dpo_acc:.1%}')

print('\nP1b 完成')

---
## P3: Group A 训练（LoRA + 单段 SFT + Standard DPO）+ 评测

- 经典 baseline，与 Group B（DoRA + 五段课程）形成消融对比
- 只跑 GSM8K（8-shot, n=300），跳过 MATH
- 断点续跑：SFT checkpoint 自动恢复
- **预计耗时：~4.5h（SFT ~2.5h + DPO ~1h + 评测 ~1h）**

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P3: Group A 训练 + 评测（LoRA baseline）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

G_A_SFT        = 'outputs/group_a/sft'
G_A_SFT_MERGED = 'outputs/group_a/sft_merged'
G_A_DPO        = 'outputs/group_a/dpo'
G_A_MERGED     = 'outputs/group_a/merged'

def run_train(cmd, label):
    ts       = datetime.now().strftime('%Y%m%d_%H%M%S')
    log_dir  = f'logs/runs/{ts}_{label}'
    os.makedirs(log_dir, exist_ok=True)
    log_path = f'{log_dir}/train.log'
    rc = stream_run(cmd, log_path=log_path)
    if rc != 0:
        raise RuntimeError(f'{label} 失败 (exit {rc})，日志: {log_path}')
    print(f'  日志: {log_path}')

# ── P3.1: Group A SFT ──────────────────────────────────────────
if Path(f'{G_A_SFT_MERGED}/config.json').exists():
    print('✅ Group A sft_merged 已存在，跳过 SFT')
else:
    print('▶ P3.1: Group A SFT（LoRA + 单段混合，~2.5h）')
    if not Path('data/processed/sft_train.json').exists():
        raise FileNotFoundError('data/processed/sft_train.json 不存在')
    sft_cfg = {
        'model_name': 'Qwen/Qwen2.5-1.5B-Instruct',
        'output_dir': G_A_SFT,
        'max_seq_length': 2048, 'load_in_4bit': True, 'seed': 42,
        'stages': [{'name': 'stage_mixed', 'dataset': {
            'path': 'data/processed/sft_train.json', 'max_samples': 15000,
            'field_map': {'prompt': 'input', 'response': 'output'}},
            'train': {'max_steps': 900, 'learning_rate': 3e-5, 'warmup_steps': 90}}],
        'lora': {'use_dora': False, 'r': 16, 'alpha': 32, 'dropout': 0.0,
                 'target_modules': ['q_proj','k_proj','v_proj','o_proj',
                                    'gate_proj','up_proj','down_proj']},
        'train': {
            'per_device_train_batch_size': BATCH_SIZE, 'gradient_accumulation_steps': 8,
            'warmup_steps': 90, 'max_steps': 900, 'learning_rate': 3e-5,
            'logging_steps': 10, 'save_steps': 300, 'weight_decay': 0.01,
            'lr_scheduler_type': 'cosine', 'optim': 'paged_adamw_8bit',
            'fp16': False, 'bf16': True, 'dataloader_num_workers': 4, 'packing': True,
        },
        'dataset_path': 'data/processed/sft_train.json',
    }
    with open('config/sft_config_group_a.yaml', 'w') as f:
        yaml.dump(sft_cfg, f, allow_unicode=True)
    ckpts = sorted(glob.glob(f'{G_A_SFT}/checkpoint-*'))
    if ckpts: print(f'  ♻️ 续跑: {ckpts[-1]}')
    run_train(['python3', 'scripts/sft_train.py',
               '--config', 'config/sft_config_group_a.yaml'], 'group_a_sft')
    os.makedirs(G_A_SFT_MERGED, exist_ok=True)
    subprocess.run(['python3', 'scripts/merge_lora.py',
                    '--adapter_path', G_A_SFT,
                    '--output_path', G_A_SFT_MERGED], check=True)
    print('  Group A SFT + merge ✓')
print('Group A SFT ✓')

# ── P3.2: Group A DPO ──────────────────────────────────────────
if Path(f'{G_A_MERGED}/config.json').exists():
    print('✅ Group A merged 已存在，跳过 DPO')
else:
    print('▶ P3.2: Group A DPO（Standard DPO，~1h）')
    if not Path('data/processed/dpo_train.json').exists():
        raise FileNotFoundError('data/processed/dpo_train.json 不存在')
    dpo_cfg = {
        'model_name': 'Qwen/Qwen2.5-1.5B-Instruct',
        'base_adapter_path': G_A_SFT_MERGED,
        'output_dir': G_A_DPO,
        'max_seq_length': 2048, 'load_in_4bit': True, 'seed': 42,
        'beta': 0.1, 'loss_type': 'sigmoid',
        'dataset': {'name': 'local', 'split': 'train', 'max_samples': 5000},
        'lora': {'use_dora': False, 'r': 16, 'alpha': 32, 'dropout': 0.0,
                 'target_modules': ['q_proj','k_proj','v_proj','o_proj',
                                    'gate_proj','up_proj','down_proj']},
        'train': {
            'per_device_train_batch_size': BATCH_SIZE, 'gradient_accumulation_steps': 8,
            'warmup_steps': 50, 'max_steps': 600, 'learning_rate': 1e-5,
            'logging_steps': 10, 'save_steps': 100, 'weight_decay': 0.0,
            'lr_scheduler_type': 'cosine', 'optim': 'paged_adamw_8bit',
            'fp16': False, 'bf16': True, 'dataloader_num_workers': 4,
        },
        'dataset_path': 'data/processed/dpo_train.json',
    }
    with open('config/dpo_config_group_a.yaml', 'w') as f:
        yaml.dump(dpo_cfg, f, allow_unicode=True)
    ckpts = sorted(glob.glob(f'{G_A_DPO}/checkpoint-*'))
    if ckpts: print(f'  ♻️ 续跑: {ckpts[-1]}')
    run_train(['python3', 'scripts/dpo_train.py',
               '--config', 'config/dpo_config_group_a.yaml'], 'group_a_dpo')
    os.makedirs(G_A_MERGED, exist_ok=True)
    subprocess.run(['python3', 'scripts/merge_lora.py',
                    '--adapter_path', G_A_DPO,
                    '--base_model', G_A_SFT_MERGED,
                    '--output_path', G_A_MERGED], check=True)
    print('  Group A DPO + merge ✓')
print('Group A DPO ✓')

# ── P3.3: Group A 评测（GSM8K only）───────────────────────────
G_A_SFT_MERGED = ensure_fp16_merged(G_A_SFT_MERGED, G_A_SFT, 'Group A SFT')
G_A_MERGED     = ensure_fp16_merged(G_A_MERGED, G_A_DPO, 'Group A DPO')

print('\n▶ P3.3: Group A 评测 (GSM8K 8-shot, n=300)')
if Path(f'{G_A_SFT_MERGED}/config.json').exists():
    print('  评测 Group A SFT...')
    run_lm_eval_gsm8k(G_A_SFT_MERGED, V1_GSM_A_SFT, limit=300)
if Path(f'{G_A_MERGED}/config.json').exists():
    print('  评测 Group A DPO...')
    run_lm_eval_gsm8k(G_A_MERGED, V1_GSM_A_DPO, limit=300)

print('\nP3 完成')

---
## P5: 汇总所有 V1 结果

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# P5: 汇总 V1 实验结果
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

def load_result(path):
    try:
        d = json.loads(Path(path).read_text())
        return d.get('accuracy'), d.get('total', 0)
    except Exception:
        return None, 0

groups = [
    ('A-SFT', 'LoRA + 单段SFT (no DPO)',          V1_GSM_A_SFT),
    ('A',     'LoRA + 单段SFT + Std DPO',          V1_GSM_A_DPO),
    ('B-SFT', 'DoRA + 五段课程 (no DPO)',           V1_GSM_B_SFT),
    ('B',     'DoRA + 五段课程 + Std DPO',          V1_GSM_B_DPO),
    ('D',     'DoRA + 五段课程 + Targeted DPO v1',  V1_GSM_D),
]

print('=' * 72)
print(f'{"Group":8} {"Description":42} {"GSM8K(8-shot)":>13} {"n":>5}')
print('-' * 72)
rows = []
for g, desc, path in groups:
    acc, n = load_result(path)
    acc_s = f'{acc*100:.1f}%' if acc is not None else 'N/A'
    n_s   = str(n) if n else '-'
    print(f'{g:8} {desc:42} {acc_s:>13} {n_s:>5}')
    rows.append({'group': g, 'desc': desc, 'gsm8k_8shot': acc, 'n': n})
print('=' * 72)

print('\n📊 Qwen 官方 baseline (lm-eval 8-shot):')
print('  Qwen2.5-1.5B (no finetuning): 73.2%')
print('  Qwen2.5-7B:                   91.6%')

print('\n📊 自定义协议补充（logs 2/, n=200, CI ±6.9pp）:')
print('  A:     GSM8K=63.5%  MATH=44.5%  BBH=38.5%')
print('  B-SFT: GSM8K=61.5%  MATH=44.0%  BBH=38.8%')
print('  B:     GSM8K=62.0%  MATH=47.5%')
print('  D:     GSM8K=64.5%  MATH=44.0%  BBH=37.4%')

os.makedirs('results/ablation', exist_ok=True)
with open('results/ablation/summary_v1.json', 'w') as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print('\n💾 结果已保存: results/ablation/summary_v1.json')
print('\n✅ 全部完成！')